# Phase A: Data Preparation for CLIP ViT-B/32 on CelebA
## Steps 1-3: Load dataset & attributes (from data_loader), load evaluation JSON, sanity checks

## Step 1: Load CelebA Dataset (test split only)

**Why:** We need the actual images in a standardized format with CLIP-compatible preprocessing.
- Resize to 224×224 (CLIP ViT-B/32's expected input)
- Normalize using CLIP's standard statistics
- Load only test split (~19,962 images) to evaluate on unseen data


In [10]:
import torch
import json
from pathlib import Path
from data_loader import load_celeba_dataset, load_attributes, ATTRIBUTE_NAMES, setup_frozen_db

# Determine project root
notebook_dir = Path.cwd()
project_root = notebook_dir.parent if notebook_dir.name == 'project' else notebook_dir

print(f"Project root: {project_root}")
print()

# Ensure frozen DB exists
setup_frozen_db()
print()

# Load CelebA dataset and attributes
celeba = load_celeba_dataset()
celeba_attrs = load_attributes()

print(f"✓ CelebA test set loaded successfully")
print(f"  Dataset size: {len(celeba)} images")
print(f"  Attributes shape: {celeba_attrs.shape}")
print(f"  Attributes dtype: {celeba_attrs.dtype}")
print()

Project root: c:\Users\alfon\Desktop\Deep-Learning-Project

✓ Frozen DB already exists: c:\Users\alfon\Desktop\Deep-Learning-Project\celeba_attributes_test.pt

✓ CelebA test set loaded successfully
  Dataset size: 19962 images
  Attributes shape: torch.Size([19962, 40])
  Attributes dtype: torch.float32



## Step 2: Load Evaluation JSON

**Why:** This JSON defines the ground truth benchmark for evaluating retrieval quality.

Each query specifies:
- **Sources:** All images matching the query condition (e.g., 4,786 smiling images for "+Smiling")
- **Valid targets:** For each source, the list of other images that are similar by attribute (Hamming distance ≤ 2) AND match the query condition

**How it's built:**
1. For each of the 4,786 smiling images (source):
   - Find all similar images using Hamming distance ≤ 2
   - Filter to only those that ALSO have "+Smiling" attribute
   - Record this list as valid targets for this source
2. Average: (targets for source1 + targets for source2 + ... ) / 4,786 = **26.0 avg targets**

**Later evaluation (Phase B):**
- Extract CLIP embeddings for all test images
- For each source, retrieve top-26 similar images using CLIP
- Check: How many of CLIP's retrievals match the ground truth targets?
- Score: Recall/Precision = (correct retrievals / expected targets)

In [ ]:
eval_json_path = project_root / 'Evaluation' / 'celeba_evaluation.json'
print(f"Reading from: {eval_json_path}")

with open(eval_json_path, 'r') as f:
    queries = json.load(f)

print(f"  Total queries: {len(queries)}")

# Display summary of each query
for i, query_dict in enumerate(queries):
    query_str = query_dict['query']
    ground_truth = query_dict['ground_truth']
    num_sources = len(ground_truth)

    # Count total targets
    total_targets = sum(len(targets) for targets in ground_truth.values())
    avg_targets = total_targets / num_sources if num_sources > 0 else 0

    print(f"  Query {i}: '{query_str}'")
    print(f"    Sources: {num_sources}, Avg targets per source: {avg_targets:.1f}")

## Step 3: Sanity Check (CRITICAL)

**Why:** Index consistency is fragile—dataset indices don't match filenames.
- Verify: `celeba[13]` corresponds to filename in JSON
- Verify: attribute tensor has correct shape and values
- Verify: ground truth structure matches expectations
- **One failure here means everything downstream is wrong**

In [ ]:
# Test index from ROADMAP
test_idx = 13

print(f"Testing with index: {test_idx}")
print()

# Check 1: Dataset metadata
target_filename = celeba.filename[test_idx]
print(f"✓ Check 1: Dataset index to filename")
print(f"  celeba.filename[{test_idx}] = {target_filename}")
print()

# Check 2: Attribute tensor indexing
attr_row = celeba_attrs[test_idx]
print(f"✓ Check 2: Attribute tensor indexing")
print(f"  celeba_attrs[{test_idx}].shape = {attr_row.shape}")
print(f"  Sample attributes (first 5): {attr_row[:5].tolist()}")
print()

# Check 3: JSON ground truth structure
query_0 = queries[0]
query_str = query_0['query']
ground_truth = query_0['ground_truth']

print(f"✓ Check 3: JSON ground truth structure")
print(f"  Query 0: '{query_str}'")

targets = ground_truth[str(test_idx)]
print(f"  Source {test_idx} found in ground truth")
print(f"  Number of valid targets: {len(targets)}")
print(f"  Sample targets (first 5): {targets[:5]}")